# Day 2: NanoLlama v3 — Build a Modern LLM from Scratch

**Duration:** ~3 hours | **GPU Time:** ~15–25 min (training)

Today you implement every component of a **51M-parameter Llama-style language model** and train it end-to-end on WikiText-103.

### Architecture upgrades over GPT-2

| GPT-2 | NanoLlama v3 | Benefit |
|-------|-------------|--------|
| Learned pos. embeddings | **RoPE** | Relative position; length generalisation |
| LayerNorm | **RMSNorm** | ~10% faster; no bias; matches Llama/Mistral |
| GELU MLP | **SwiGLU** | Gated FFN; better loss-per-param |
| Full MHA (8Q/8KV) | **GQA (8Q/4KV)** | 2× KV memory savings at inference |
| Biased linears | **No bias** | Regularisation; standard in modern LLMs |

## 0  Setup (re-run Day 1 pipeline)

Each Kaggle notebook runs in an isolated kernel. We re-execute the Day 1 data pipeline in one cell to keep Day 2 self-contained.

In [ ]:
import os
import math
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset

print(f'PyTorch: {torch.__version__}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_gpus = torch.cuda.device_count()
print(f'Device: {device} | GPUs: {n_gpus}')

torch.manual_seed(42)
if device == 'cuda':
    torch.cuda.manual_seed_all(42)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    for i in range(n_gpus):
        mem_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem_gb:.1f} GB)')

# ── Tokenizer ────────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained('gpt2')
vocab_size = tokenizer.vocab_size
print(f'\nTokenizer: GPT-2 BPE | Vocab size: {vocab_size:,}')

# ── WikiText-103 ─────────────────────────────────────────
print('\nLoading WikiText-103...')
raw_dataset = load_dataset('wikitext', 'wikitext-103-raw-v1')

print('Tokenizing train split...')
train_lines = [line for line in raw_dataset['train']['text'] if line.strip()]
train_ids   = []
chunk_size  = 5000
for i in range(0, len(train_lines), chunk_size):
    chunk = '\n'.join(train_lines[i : i + chunk_size])
    train_ids.extend(tokenizer.encode(chunk))
    if i % 50000 == 0:
        print(f'  {i:>7,} / {len(train_lines):,} lines...')
print(f'  {len(train_ids):,} train tokens')

print('Tokenizing val split...')
val_lines = [line for line in raw_dataset['validation']['text'] if line.strip()]
val_ids   = tokenizer.encode('\n'.join(val_lines))
print(f'  {len(val_ids):,} val tokens')

train_data = torch.tensor(train_ids, dtype=torch.long)
val_data   = torch.tensor(val_ids,   dtype=torch.long)

del raw_dataset, train_lines, val_lines, train_ids, val_ids
torch.cuda.empty_cache()

# ── SlidingWindowDataset ─────────────────────────────────
class SlidingWindowDataset(Dataset):
    def __init__(self, tokens, block_size=1024, stride=512):
        self.tokens     = tokens
        self.block_size = block_size
        self.starts     = list(range(0, len(tokens) - block_size - 1, stride))
    def __len__(self):
        return len(self.starts)
    def __getitem__(self, idx):
        s = self.starts[idx]
        return self.tokens[s : s + self.block_size], self.tokens[s + 1 : s + self.block_size + 1]

print('\nSetup complete.')

## 1  Model Configuration

All hyperparameters live in a single `LlamaConfig` dataclass. This mirrors the HuggingFace `PretrainedConfig` pattern — one object you pass around everywhere, easy to serialise to JSON alongside a checkpoint.

### Parameter budget (~51M)

| Component | Calculation | Params |
|-----------|-------------|--------|
| Token embedding | 50,257 × 512 | 25.7M |
| GQA per block | 8Q+4K+4V+output projections | ~786K |
| SwiGLU per block | gate+up+down | ~2.4M |
| 8 blocks total | (786K + 2.4M) × 8 | 25.5M |
| RMSNorm + tied head | ≈ 0M (weight tied) | ~0M |
| **Total** | | **≈ 51.2M** |

**Weight tying:** the LM head reuses the embedding matrix — no extra parameters for the output projection.

In [ ]:
@dataclass
class LlamaConfig:
    vocab_size:        int   = 50257
    block_size:        int   = 1024    # context window (tokens)
    n_layer:           int   = 8       # transformer blocks
    n_head:            int   = 8       # query heads
    n_kv_head:         int   = 4       # key-value heads (GQA)
    n_embd:            int   = 512     # embedding / hidden dimension
    intermediate_size: int   = 1536    # SwiGLU expansion
    dropout:           float = 0.1
    rope_theta:        float = 10000.0

config = LlamaConfig(vocab_size=vocab_size)
print(config)

## 2  RMSNorm

Standard **LayerNorm** computes:
$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma} \cdot \gamma + \beta$$

**RMSNorm** (Zhang & Sennrich, 2019) drops the mean-centering step:
$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma \qquad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_i x_i^2}$$

**Why?**
- Mean-centering adds computation but empirically contributes little to quality.
- No bias vector $\beta$ — one fewer parameter per norm layer.
- Used in Llama 1/2/3, Mistral, Qwen, Gemma, DeepSeek.

**Implementation note:** cast to `float32` before squaring to prevent overflow in `float16` activations, then cast back.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps    = eps

    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).type_as(x) * self.weight


# ── Verify ────────────────────────────────────────────────
rms   = RMSNorm(config.n_embd)
ln    = nn.LayerNorm(config.n_embd)
x     = torch.randn(2, 8, config.n_embd) * 5   # simulate unnormalised activations

y_rms = rms(x)
y_ln  = ln(x)

print(f'Input  x  : mean={x.mean():.3f}   std={x.std():.3f}')
print(f'RMSNorm y : mean={y_rms.mean():.3f}   std={y_rms.std():.3f}')
print(f'LayerNorm y: mean={y_ln.mean():.3f}   std={y_ln.std():.3f}')
print()

rms_params = sum(p.numel() for p in rms.parameters())
ln_params  = sum(p.numel() for p in ln.parameters())
print(f'RMSNorm params : {rms_params}  (gamma only)')
print(f'LayerNorm params: {ln_params}  (gamma + beta)')

## 3  RoPE: Rotary Position Embeddings

GPT-2 adds a learned position embedding to token embeddings before the first block. This couples position information to the token representation and requires re-training when the context length changes.

**RoPE** (Su et al., 2021) instead applies a *rotation* directly to query and key vectors inside each attention layer:

$$q_m' = q_m \cdot e^{im\theta} \qquad k_n' = k_n \cdot e^{in\theta}$$

The dot product $q_m'^\top k_n'$ then depends only on the *relative* distance $(m - n)$, not absolute positions. This gives:
- **Relative position encoding** naturally, without extra parameters.
- **Length generalisation** — attention patterns are shift-invariant.
- **No learned position table** — zero extra parameters.

### Implementation

For real-valued vectors we work dimension-pair-wise. For a vector split into two halves $[x_1, x_2]$:

$$\text{rotate\_half}(x) = [-x_2,\; x_1]$$
$$x' = x \cos(m\theta) + \text{rotate\_half}(x) \sin(m\theta)$$

Frequencies $\theta_i = \text{rope\_theta}^{-2i/d}$ decrease geometrically, encoding fine-grained position at low-index dimensions and coarse structure at high-index dimensions.

In [ ]:
def rotate_half(x):
    """Swap the two halves of the last dimension and negate the first."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_emb(q, k, cos, sin):
    """Apply RoPE to query and key tensors.
    cos, sin : (T, head_dim)  —  broadcast over batch and head dims.
    """
    cos = cos.unsqueeze(0).unsqueeze(0)   # (1, 1, T, head_dim)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


# ── Precompute sin/cos table (done once in NanoLlama.__init__) ──────
head_dim = config.n_embd // config.n_head   # 512 / 8 = 64
freqs    = 1.0 / (config.rope_theta ** (
    torch.arange(0, head_dim, 2).float() / head_dim
))                                           # (head_dim/2,)
t        = torch.arange(config.block_size, dtype=torch.float32)
freqs    = torch.outer(t, freqs)             # (block_size, head_dim/2)

rope_cos = torch.cat([freqs.cos(), freqs.cos()], dim=-1)   # (T, head_dim)
rope_sin = torch.cat([freqs.sin(), freqs.sin()], dim=-1)

print(f'head_dim : {head_dim}')
print(f'rope_cos : {rope_cos.shape}   (block_size={config.block_size}, head_dim={head_dim})')
print()

# Demo: apply RoPE to random Q, K
B, H, T = 1, config.n_head, 16
q = torch.randn(B, H, T, head_dim)
k = torch.randn(B, H, T, head_dim)

cos_slice = rope_cos[:T]
sin_slice = rope_sin[:T]
q_rot, k_rot = apply_rotary_emb(q, k, cos_slice, sin_slice)

print(f'q shape     : {q.shape}')
print(f'q_rot shape : {q_rot.shape}  (same — rotation preserves shape)')
print(f'Norms preserved: q_norm={q.norm(dim=-1).mean():.3f}  q_rot_norm={q_rot.norm(dim=-1).mean():.3f}')
print('Rotation preserves vector norms — only direction changes, not magnitude.')

## 4  Grouped Query Attention (GQA)

Standard **Multi-Head Attention (MHA)** has equal numbers of Q, K, V heads. At inference, the K and V tensors must be cached for each layer and each head — this is the **KV-cache**, which dominates memory at long context lengths.

**GQA** (Ainslie et al., 2023) uses fewer KV heads than Q heads. Multiple Q heads share a single KV head pair:

```
MHA (8Q / 8KV):   Q₀K₀V₀  Q₁K₁V₁  ...  Q₇K₇V₇   ← 8 independent KV pairs
GQA (8Q / 4KV):   Q₀Q₁ share K₀V₀    Q₂Q₃ share K₁V₁   ...  ← 4 KV pairs
```

NanoLlama uses 8Q / 4KV (n_rep = 2). This halves KV-cache memory with negligible quality loss at 51M scale.

`repeat_kv` expands the 4 KV heads back to 8 before the attention dot-product, using a **view + expand** trick (no memory copy).

In [ ]:
def repeat_kv(x, n_rep):
    """Expand KV heads to match Q head count without copying memory.
    (B, n_kv_head, T, D) -> (B, n_head, T, D)
    """
    if n_rep == 1:
        return x
    B, n_kv, T, D = x.shape
    x = x[:, :, None, :, :].expand(B, n_kv, n_rep, T, D)
    return x.reshape(B, n_kv * n_rep, T, D)


class GQAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head    = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim  = config.n_embd // config.n_head
        self.n_rep     = config.n_head // config.n_kv_head

        # Separate Q / KV projections — bias=False (Llama convention)
        self.q_proj = nn.Linear(config.n_embd, config.n_head    * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_embd, config.n_embd,                    bias=False)

        self.attn_drop  = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)

        # Causal mask — precomputed once, reused every forward
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(config.block_size, config.block_size))
                  .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x, cos, sin):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head,    self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rotary_emb(q, k, cos, sin)     # inject position via RoPE
        k    = repeat_kv(k, self.n_rep)              # (B, n_head, T, head_dim)
        v    = repeat_kv(v, self.n_rep)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.o_proj(y))


# ── Verify ────────────────────────────────────────────────
gqa = GQAttention(config)
T   = 16
x   = torch.randn(2, T, config.n_embd)
cos_s = rope_cos[:T]
sin_s = rope_sin[:T]

with torch.no_grad():
    y = gqa(x, cos_s, sin_s)

gqa_params = sum(p.numel() for p in gqa.parameters())
print(f'GQAttention params : {gqa_params:,}')
print(f'  Q proj  : {config.n_embd} x {config.n_head * (config.n_embd // config.n_head)} = {config.n_embd * config.n_head * (config.n_embd // config.n_head):,}')
print(f'  KV proj : {config.n_embd} x {config.n_kv_head * (config.n_embd // config.n_head)} x2 = {config.n_embd * config.n_kv_head * (config.n_embd // config.n_head) * 2:,}')
print(f'  O proj  : {config.n_embd} x {config.n_embd} = {config.n_embd ** 2:,}')
print(f'Input  : {x.shape}')
print(f'Output : {y.shape}')

## 5  SwiGLU: Gated Feed-Forward Network

The feed-forward sublayer in GPT-2 is a simple two-layer MLP with GELU:
$$\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 x)$$

**SwiGLU** (Noam Shazeer, 2020) replaces this with a **gated** variant:
$$\text{SwiGLU}(x) = W_{\text{down}} \cdot (\text{SiLU}(W_{\text{gate}}\, x) \odot W_{\text{up}}\, x)$$

- **SiLU(x) = x · σ(x)** — smooth, non-monotone activation with better gradients than ReLU.
- The gate $\text{SiLU}(W_{\text{gate}} x)$ modulates how much of the $W_{\text{up}} x$ signal passes through.
- Three matrices instead of two, but `intermediate_size` is set smaller (1536 vs 2048) to keep total params similar.

SwiGLU consistently outperforms GELU FFN at the same parameter budget. Used in Llama 1/2/3, Mistral, Qwen, Gemma, PaLM.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden = config.intermediate_size

        self.gate_proj = nn.Linear(config.n_embd, hidden, bias=False)
        self.up_proj   = nn.Linear(config.n_embd, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, config.n_embd, bias=False)
        self.drop      = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.drop(
            self.down_proj(
                F.silu(self.gate_proj(x)) * self.up_proj(x)
            )
        )


ffn = SwiGLU(config)
x   = torch.randn(2, 8, config.n_embd)
y   = ffn(x)

ffn_params = sum(p.numel() for p in ffn.parameters())
print(f'SwiGLU params     : {ffn_params:,}')
print(f'  gate_proj       : {config.n_embd} x {config.intermediate_size} = {config.n_embd * config.intermediate_size:,}')
print(f'  up_proj         : {config.n_embd} x {config.intermediate_size} = {config.n_embd * config.intermediate_size:,}')
print(f'  down_proj       : {config.intermediate_size} x {config.n_embd} = {config.intermediate_size * config.n_embd:,}')
print(f'Input  : {x.shape}')
print(f'Output : {y.shape}')

# Compare SiLU vs GELU vs ReLU
t    = torch.linspace(-3, 3, 7)
print(f'\nActivation comparison at x = {t.tolist()}')
print(f'  SiLU : {F.silu(t).round(decimals=3).tolist()}')
print(f'  GELU : {F.gelu(t).round(decimals=3).tolist()}')
print(f'  ReLU : {F.relu(t).round(decimals=3).tolist()}')

## 6  Transformer Block

A transformer block applies attention and FFN in sequence, each with a residual connection. The **pre-norm** variant (used in NanoLlama, Llama, GPT-NeoX) normalises the input *before* the sublayer:

```
x  ──┬──> RMSNorm ──> GQAttention ──> + ──> x'
     └────────────────────────────────┘

x' ──┬──> RMSNorm ──> SwiGLU ──────> + ──> output
     └──────────────────────────────┘
```

**Why pre-norm?**
- Post-norm (original Transformer) can produce large gradients at early layers, requiring careful initialisation and learning rate warmup.
- Pre-norm keeps the residual stream clean — the skip connection always carries the original signal forward, preventing vanishing gradients in deep networks.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.n_embd)
        self.attn      = GQAttention(config)
        self.ffn_norm  = RMSNorm(config.n_embd)
        self.ffn       = SwiGLU(config)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)   # pre-norm + residual
        x = x + self.ffn(self.ffn_norm(x))               # pre-norm + residual
        return x


block = TransformerBlock(config)
x     = torch.randn(2, 16, config.n_embd)

with torch.no_grad():
    y = block(x, cos_s, sin_s)

block_params = sum(p.numel() for p in block.parameters())
attn_params  = sum(p.numel() for p in block.attn.parameters())
ffn_params   = sum(p.numel() for p in block.ffn.parameters())
norm_params  = sum(p.numel() for p in block.attn_norm.parameters()) + \
               sum(p.numel() for p in block.ffn_norm.parameters())

print(f'Block params total : {block_params:,}')
print(f'  GQAttention      : {attn_params:,}  ({attn_params/block_params:.0%})')
print(f'  SwiGLU           : {ffn_params:,}  ({ffn_params/block_params:.0%})')
print(f'  RMSNorm x2       : {norm_params:,}')
print(f'Input  : {x.shape}')
print(f'Output : {y.shape}  (shape preserved through block)')

## 7  NanoLlama: Full Model

The full model stacks 8 `TransformerBlock`s between an embedding lookup and an output head.

### Weight tying
```python
self.head.weight = self.tok_emb.weight
```
The output projection shares the embedding matrix. This:
- Saves 25.7M parameters (biggest single saving in the model).
- Forces the model to produce output logits in the same space as input embeddings, which improves generalisation.

### Weight initialisation
- Linear weights: `N(0, 0.02)` — same as GPT-2.
- Embedding weights: `N(0, 0.02)`.
- No bias anywhere (`bias=False` in all linears).

### Autoregressive generation
During inference we sample one token at a time, appending it to the context and re-running the model. Two sampling controls:
- **Temperature** — divide logits by T before softmax. T→0 makes output deterministic (argmax); T→∞ makes it uniform random.
- **Top-k** — zero out all logits except the top-k before sampling. Prevents the model from sampling very low-probability tokens.

In [ ]:
class NanoLlama(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop    = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layer)
        ])

        self.norm = RMSNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight   # weight tying

        # Precompute RoPE tables — registered as buffers so they move
        # to the correct device automatically with .to(device)
        head_dim = config.n_embd // config.n_head
        freqs    = 1.0 / (config.rope_theta ** (
            torch.arange(0, head_dim, 2).float() / head_dim
        ))
        t     = torch.arange(config.block_size, dtype=torch.float32)
        freqs = torch.outer(t, freqs)            # (block_size, head_dim/2)
        self.register_buffer('rope_cos', torch.cat([freqs.cos(), freqs.cos()], dim=-1))
        self.register_buffer('rope_sin', torch.cat([freqs.sin(), freqs.sin()], dim=-1))

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x    = self.drop(self.tok_emb(idx))

        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]

        for block in self.blocks:
            x = block(x, cos, sin)

        logits = self.head(self.norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8, top_k=50):
        """Autoregressive generation with sliding context window."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs   = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx     = torch.cat([idx, next_id], dim=1)
        return idx

## 8  Build the Model

In [ ]:
model     = NanoLlama(config).to(device)
raw_model = model   # keep unwrapped reference for generation

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'{"="*60}')
print(f' NanoLlama v3 — Architecture Summary')
print(f'{"="*60}')
print(f'  Params:           {total_params / 1e6:.1f}M ({train_params / 1e6:.1f}M trainable)')
print(f'  Layers:           {config.n_layer}')
print(f'  Heads (Q/KV):     {config.n_head} / {config.n_kv_head}  (GQA ratio {config.n_head // config.n_kv_head}:1)')
print(f'  Embedding dim:    {config.n_embd}')
print(f'  FFN hidden dim:   {config.intermediate_size}')
print(f'  Context window:   {config.block_size}')
print(f'  Attention:        GQA + RoPE')
print(f'  Normalization:    RMSNorm (pre-norm)')
print(f'  FFN:              SwiGLU')
print(f'  Bias:             None')
print(f'  Weight tying:     tok_emb == head')
print(f'{"="*60}')

if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f'  DataParallel across {n_gpus} GPUs')

# Quick forward pass sanity check
ids = torch.randint(0, config.vocab_size, (2, 32)).to(device)
with torch.no_grad():
    logits, _ = model(ids)
print(f'\nForward pass: ids {ids.shape} -> logits {logits.shape}  [OK]')

## 9  Training Recipe

### Hyperparameters

| Setting | Value | Why |
|---------|-------|-----|
| Batch size | 16 (2×T4) / 8 (1 GPU) | Fill GPU memory |
| Max steps | 3,000 | ~15–25 min on 2×T4 |
| Max LR | 3e-4 | Standard for Adam on transformers |
| Min LR | 3e-5 | 10% of max; don't decay to zero |
| Warmup | 200 steps | Linear ramp prevents early instability |
| Weight decay | 0.1 | L2 regularisation on weights only |
| Grad clip | 1.0 | Prevents rare explosive gradient steps |
| Betas | (0.9, 0.95) | 0.95 instead of 0.999 for noisier gradients |

### LR schedule: linear warmup → cosine decay

$$\text{lr}(t) = \begin{cases} \text{lr}_{\max} \cdot t / t_{\text{warm}} & t < t_{\text{warm}} \\ \text{lr}_{\min} + \frac{1}{2}(\text{lr}_{\max} - \text{lr}_{\min})(1 + \cos(\pi \cdot r)) & \text{otherwise} \end{cases}$$

where $r = (t - t_{\text{warm}}) / (t_{\max} - t_{\text{warm}})$.

### Automatic Mixed Precision (AMP)

AMP runs forward/backward in FP16 (faster, less memory) while keeping a master copy of weights in FP32. `GradScaler` multiplies the loss by a large constant before backward, then divides the gradients back — this prevents FP16 underflow in small gradients.

In [ ]:
block_size    = config.block_size
batch_size    = 16 if n_gpus >= 2 else 8
max_steps     = 3000
eval_interval = 200
eval_iters    = 20

max_lr       = 3e-4
min_lr       = 3e-5
warmup_steps = 200
weight_decay = 0.1
grad_clip    = 1.0

# ── Datasets & DataLoaders ───────────────────────────────
train_ds = SlidingWindowDataset(train_data, block_size=block_size, stride=512)
val_ds   = SlidingWindowDataset(val_data,   block_size=block_size, stride=512)

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    drop_last=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False,
    drop_last=True, num_workers=2, pin_memory=True
)
print(f'Train: {len(train_ds):,} windows  |  Val: {len(val_ds):,} windows')
print(f'Steps per epoch: ~{len(train_ds) // batch_size:,}')

# ── Optimizer ────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=max_lr,
    betas=(0.9, 0.95),
    weight_decay=weight_decay
)

# ── LR schedule ──────────────────────────────────────────
def get_lr(step):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    ratio = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * ratio))

# ── AMP ──────────────────────────────────────────────────
use_amp = (device == 'cuda')
try:
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    def autocast_ctx(): return torch.amp.autocast('cuda', enabled=use_amp)
    print('AMP: torch.amp')
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    def autocast_ctx(): return torch.cuda.amp.autocast(enabled=use_amp)
    print('AMP: torch.cuda.amp (legacy)')

## 10  Evaluation

We estimate loss on a fixed number of batches (`eval_iters=20`) from each split rather than the full dataset — this is fast enough to run every 200 steps without dominating training time.

**Perplexity (PPL)** = $e^{\text{loss}}$ — an intuitive metric: PPL measures the average number of equally-likely choices the model considers at each token. Random guessing over 50,257 vocab = PPL 50,257. A good model on WikiText achieves PPL < 100 after modest training.

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split_name, loader in [('train', train_loader), ('val', val_loader)]:
        losses    = []
        data_iter = iter(loader)
        for _ in range(eval_iters):
            try:
                xb, yb = next(data_iter)
            except StopIteration:
                break
            xb, yb = xb.to(device), yb.to(device)
            with autocast_ctx():
                _, loss = model(xb, yb)
            losses.append(loss.mean().item())   # .mean() for DataParallel
        out[split_name] = sum(losses) / max(len(losses), 1)
    model.train()
    return out

## 11  Training Loop

The loop follows the standard recipe for transformer training:

```
for each step:
    1. Set LR for this step (warmup / cosine)
    2. Forward pass under AMP autocast
    3. Scale loss, backward
    4. Unscale gradients, clip norm to 1.0
    5. Optimizer step, scaler update
    6. Every eval_interval steps: estimate loss, log PPL
```

**Gradient clipping** (`clip_grad_norm_`) rescales the global gradient vector if its L2 norm exceeds 1.0. This prevents occasional explosive updates that would otherwise destabilise training.

In [ ]:
print(f'{"="*60}')
print(f' TRAINING: {max_steps} steps | batch {batch_size} | ctx {block_size}')
print(f' NanoLlama v3: RoPE + RMSNorm + SwiGLU + GQA')
print(f'{"="*60}\n')

start_time    = time.time()
model.train()
step          = 0
best_val_loss = float('inf')

while step < max_steps:
    for xb, yb in train_loader:
        if step >= max_steps:
            break

        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast_ctx():
            _, loss = model(xb, yb)
        loss = loss.mean()   # DataParallel returns one loss per GPU

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()

        if step % eval_interval == 0:
            losses  = estimate_loss()
            elapsed = (time.time() - start_time) / 60
            ppl     = math.exp(min(losses['val'], 20))
            marker  = ''
            if losses['val'] < best_val_loss:
                best_val_loss = losses['val']
                marker = ' *'
            print(
                f'step {step:5d} | '
                f'train {losses["train"]:.4f} | '
                f'val {losses["val"]:.4f} | '
                f'ppl {ppl:7.1f} | '
                f'lr {lr:.2e} | '
                f'{elapsed:5.1f} min{marker}'
            )
        step += 1

total_time  = (time.time() - start_time) / 60
final_loss  = estimate_loss()
final_ppl   = math.exp(min(final_loss['val'], 20))

print(f'\n{"="*60}')
print(f' TRAINING COMPLETE')
print(f'    Time            : {total_time:.1f} minutes')
print(f'    Final val loss  : {final_loss["val"]:.4f}')
print(f'    Final val PPL   : {final_ppl:.1f}')
print(f'    Best val loss   : {best_val_loss:.4f}')
print(f'{"="*60}')

## 12  Save Checkpoint

We save `raw_model.state_dict()` (the unwrapped model, not the `DataParallel` wrapper) so the checkpoint loads cleanly without DataParallel on a single GPU.

In [ ]:
save_path = '/kaggle/working/nanollama_v3_51m.pt'

torch.save({
    'model_state_dict': raw_model.state_dict(),
    'config'          : config,
    'best_val_loss'   : best_val_loss,
    'step'            : step,
}, save_path)

file_size_mb = os.path.getsize(save_path) / 1e6
print(f'Checkpoint saved: {save_path}  ({file_size_mb:.1f} MB)')

# Load verification
ckpt = torch.load(save_path, map_location='cpu')
print(f'Keys in checkpoint: {list(ckpt.keys())}')
print(f'Best val loss: {ckpt["best_val_loss"]:.4f}  |  Step: {ckpt["step"]}')

## 13  Text Generation

### Sampling controls

**Temperature** $T$ scales logits before softmax:
- $T < 1$ → sharper distribution → model sticks to high-probability tokens (more predictable)
- $T > 1$ → flatter distribution → model takes more creative risks (more surprising)
- $T = 0$ → argmax (greedy decoding)

**Top-k sampling** zeroes out all tokens except the top-k most likely before sampling. This prevents sampling from the long tail of near-zero probability tokens, which can produce incoherent text.

Typical defaults: `temperature=0.8, top_k=50` — creative but coherent.

In [ ]:
prompts = [
    'The history of artificial intelligence began',
    'In the beginning, there was nothing but darkness and',
    'The scientist carefully examined the results and concluded that',
    'Once upon a time in a kingdom far away,',
]

raw_model.eval()
raw_model.to(device)

print(f'{"="*60}')
print(f' TEXT GENERATION  (temperature=0.8, top_k=50)')
print(f'{"="*60}')

for prompt in prompts:
    input_ids = tokenizer.encode(prompt)
    x         = torch.tensor([input_ids], dtype=torch.long).to(device)

    output_ids = raw_model.generate(x, max_new_tokens=150, temperature=0.8, top_k=50)
    text       = tokenizer.decode(output_ids[0].tolist())

    print(f'\n{"─"*60}')
    print(f'PROMPT: {prompt!r}')
    print(f'{"─"*60}')
    print(text)

print(f'\n{"="*60}')
print(f' DONE — NanoLlama v3')
print(f'{"="*60}')

## 14  Day 2 Summary

| Component | Class | Key insight |
|-----------|-------|-------------|
| Normalisation | `RMSNorm` | Drop mean-centering; no bias; faster |
| Position | `apply_rotary_emb` | Rotation encodes relative position in Q·K dot product |
| Attention | `GQAttention` | GQA halves KV memory; RoPE replaces position tables |
| FFN | `SwiGLU` | Gate controls information flow; better loss/param than GELU |
| Block | `TransformerBlock` | Pre-norm + dual residuals for stable deep training |
| Model | `NanoLlama` | Weight tying, precomputed RoPE buffers, generate() |
| Training | AdamW + AMP | Mixed precision, cosine LR, grad clip |

### Day 3 preview: Fine-tuning Ops

Tomorrow you apply **LoRA** (Low-Rank Adaptation) to fine-tune Gemma 2b model on a downstream task.